# XGBoost Regression for IMDb Review Ratings

This notebook reads the regression data from the `processed_for_regression` folder, cleans the review text, performs the preprocessing required for XGBoost, trains a regressor, and saves cleaned CSV files in the `artifacts` folder.

### Preprocessing check
Yes — XGBoost requires preprocessing. Raw text cannot be used directly, so we clean the text and convert it into TF-IDF features before training.

In [ ]:
import os
import re
import html

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

PROJECT_ROOT = os.path.abspath('../..')
DATA_DIR = os.path.join(PROJECT_ROOT, 'processed_for_regression')
ARTIFACTS_DIR = os.path.join(os.getcwd(), 'artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

train_df = pd.read_csv(os.path.join(DATA_DIR, 'imdb_regression_train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'imdb_regression_test.csv'))

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(train_df.head(2).to_string(index=False))

---

## Step 1: Clean the text

We remove HTML tags, decode entities, normalize whitespace, and drop duplicate text records before training.

In [ ]:
def clean_text(text):
    text = html.unescape(str(text)) if pd.notna(text) else ''
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text_clean'] = train_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

train_clean = train_df.drop_duplicates(subset=['text_clean'], keep='first').copy()
test_clean = test_df.drop_duplicates(subset=['text_clean'], keep='first').copy()

train_clean = train_clean[['text_clean', 'rating']].rename(columns={'text_clean': 'text'})
test_clean = test_clean[['text_clean', 'rating']].rename(columns={'text_clean': 'text'})

train_clean.to_csv(os.path.join(ARTIFACTS_DIR, 'xgboost_regression_train_clean.csv'), index=False)
test_clean.to_csv(os.path.join(ARTIFACTS_DIR, 'xgboost_regression_test_clean.csv'), index=False)

print(f'Train rows after cleaning: {len(train_clean)}')
print(f'Test rows after cleaning: {len(test_clean)}')

---

## Step 2: TF-IDF features + XGBoost regressor

The cleaned review text is converted to numeric TF-IDF features before fitting an XGBoost regression model.

In [ ]:
train_clean = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'xgboost_regression_train_clean.csv'))
test_clean = pd.read_csv(os.path.join(ARTIFACTS_DIR, 'xgboost_regression_test_clean.csv'))

X_train, X_val, y_train, y_val = train_test_split(
    train_clean['text'].astype(str),
    train_clean['rating'].astype(float),
    test_size=0.10,
    random_state=42,
    stratify=np.round(train_clean['rating']).astype(int),
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_features=2000,
    strip_accents='unicode',
)

X_tr = vectorizer.fit_transform(X_train)
X_va = vectorizer.transform(X_val)
X_te = vectorizer.transform(test_clean['text'].astype(str))

model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=80,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='mae',
)

model.fit(X_tr, y_train)

val_pred = model.predict(X_va)
test_pred = model.predict(X_te)

print(f'Validation MAE: {mean_absolute_error(y_val, val_pred):.4f}')
print(f'Test MAE: {mean_absolute_error(test_clean['rating'].astype(float), test_pred):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(test_clean['rating'].astype(float), test_pred)):.4f}')
print(f'Test R^2: {r2_score(test_clean['rating'].astype(float), test_pred):.4f}')

---

## Step 3: Save metrics

Validation and test metrics are saved to the artifacts folder.

In [ ]:
metrics_df = pd.DataFrame({
    'set': ['validation', 'test'],
    'mae': [
        mean_absolute_error(y_val, val_pred),
        mean_absolute_error(test_clean['rating'].astype(float), test_pred),
    ],
    'rmse': [
        np.sqrt(mean_squared_error(y_val, val_pred)),
        np.sqrt(mean_squared_error(test_clean['rating'].astype(float), test_pred)),
    ],
    'r2': [
        r2_score(y_val, val_pred),
        r2_score(test_clean['rating'].astype(float), test_pred),
    ],
})

metrics_path = os.path.join(ARTIFACTS_DIR, 'metrics', 'xgboost_regression_metrics.csv')
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
metrics_df.to_csv(metrics_path, index=False)

print(metrics_df.to_string(index=False))